In [ ]:
import os
import random
import sys
import copy
import subprocess

os.environ['PYTHONUNBUFFERED'] = '1'
print('Script starting')

# =============================================================================
# MPI SETUP - Multi-node data parallelism
# =============================================================================

from mpi4py import MPI

comm = MPI.COMM_WORLD
size = comm.Get_size()
rank = comm.Get_rank()

print(f'MPI initialized: Rank {rank}/{size}')

import torch
import torch.nn as nn
import torch.nn.functional as F
from tqdm import tqdm
from torch.utils.data import TensorDataset, DataLoader
import pandas as pd
import numpy as np
import shap
import matplotlib
matplotlib.use('Agg')

state   = str(sys.argv[1])
parting = str(sys.argv[2])

print('Script imports done')

from datetime import datetime
from scipy.stats import spearmanr

# =============================================================================
# HYPERPARAMETERS
# =============================================================================

HIDDEN       = 256    # N_a and N_d (shared and step dimensions)
N_STEPS      = 6      # number of sequential attention steps
N_SHARED     = 2      # shared GLU layers per step
N_INDEP      = 2      # step-independent GLU layers per step
DROPOUT      = 0.1    # dropout on final representation
MOMENTUM     = 0.02   # BatchNorm momentum (TabNet paper uses 0.02)
GAMMA        = 1.5    # feature reusage coefficient (1.0 = no reuse penalty)
LAMBDA_SPARSE = 1e-4  # sparsity regularization weight on attention entropy
MAX_LR       = 3e-4
WEIGHT_DECAY = 0
HUBER_BETA   = 0.2

# =============================================================================
# GPU ASSIGNMENT
# =============================================================================

def get_num_gpus():
    """
    Detect number of GPUs visible to this rank.

    torch.cuda.device_count() respects CUDA_VISIBLE_DEVICES set by SLURM
    when --gpus-per-task=1 is used. nvidia-smi -L is a fallback only and
    ignores CUDA_VISIBLE_DEVICES, so it would return the full DGX node count.
    """
    n = torch.cuda.device_count()
    if n > 0:
        print(f'[Rank {rank}] torch sees {n} GPU(s)')
        return n
    try:
        output = subprocess.check_output(['nvidia-smi', '-L'],
                                         stderr=subprocess.DEVNULL)
        n = len(output.decode().strip().split('\n'))
        print(f'[Rank {rank}] nvidia-smi sees {n} GPU(s)')
        return n
    except Exception:
        print(f'[Rank {rank}] GPU detection failed, defaulting to 1')
        return 1


_num_gpus  = get_num_gpus()
local_rank = rank % _num_gpus
torch.cuda.set_device(local_rank)
device     = torch.device(f'cuda:{local_rank}')

print(f'Rank {rank}: Using device {device} '
      f'(local_rank={local_rank}, gpus_visible={_num_gpus})')
print(f'Rank {rank}: GPU name: {torch.cuda.get_device_name(device)}')

# =============================================================================
# REPRODUCIBILITY
# =============================================================================

SEED = 1337 + rank
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)

torch.backends.cuda.matmul.allow_tf32 = True
torch.backends.cudnn.allow_tf32 = True
g = torch.Generator()
g.manual_seed(SEED)

# =============================================================================
# TRAINING CONFIG
# =============================================================================

NUM_EPOCHS    = 60
WARMUP_EPOCHS = 12
BATCH_SIZE    = 4096
PATIENCE      = 12
VAL_FRAC      = 0.10

# =============================================================================
# FEATURE GROUP INDICES
# var2 = ['Clay','Sand','Silt','Elevation','Aspect','Slope',   <- static  [0-5]
#          'LAI','MODIS','ALB','Temp','LandC',                 <- dynamic [6-10]
#          'Month','Year']                                      <- temporal[11-12]
# =============================================================================

STATIC_IDX   = [0, 1, 2, 3, 4, 5]
DYNAMIC_IDX  = [6, 7, 8, 9, 10]
TEMPORAL_IDX = [11, 12]

# =============================================================================
# HELPER FUNCTIONS
# =============================================================================

def rank_corr_ave(data, verifier, subject):
    """Spearman correlation on spatially-averaged monthly anomalies."""
    data = data.dropna().copy()

    month_ave = (data[[verifier, subject, 'Year', 'Month', 'PageName']]
                 .groupby(['Year', 'Month', 'PageName']).mean()
                 .reset_index())

    hist_month_ave = (month_ave[[verifier, subject, 'Month', 'PageName']]
                      .groupby(['Month', 'PageName']).mean()
                      .reset_index()
                      .rename(columns={verifier: verifier + '_M',
                                       subject:  subject  + '_M'}))

    data_t = data.merge(hist_month_ave, on=['PageName', 'Month'])
    data_t[verifier + '_A'] = data_t[verifier] - data_t[verifier + '_M']
    data_t[subject  + '_A'] = data_t[subject]  - data_t[subject  + '_M']

    d = (data_t.groupby(['Month', 'Year'])
               .agg(verifier_A=(verifier + '_A', 'mean'),
                    subject_A =(subject  + '_A', 'mean'))
               .reset_index())

    spearman_corr, spearman_p = spearmanr(d['verifier_A'], d['subject_A'])
    return spearman_corr, spearman_p


class TabNetInferenceWrapper(nn.Module):
    """Strips the entropy term from TabNet forward for SHAP compatibility.

    shap.GradientExplainer requires a model that returns a single tensor.
    TabNet.forward returns (pred, entropy); this wrapper discards entropy
    so GradientExplainer sees only the prediction tensor.
    """
    def __init__(self, model: nn.Module):
        super().__init__()
        self.model = model

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        pred, _ = self.model(x)
        return pred


def compute_shap_values(model, X_tr, X_ts, device):
    """SHAP GradientExplainer on a trained PyTorch model."""
    model.eval()
    wrapped     = TabNetInferenceWrapper(model)
    background  = torch.tensor(X_tr[:500], dtype=torch.float32, device=device)
    test_tensor = torch.tensor(X_ts[:500], dtype=torch.float32, device=device)
    explainer   = shap.GradientExplainer(wrapped, background)
    vals        = explainer.shap_values(test_tensor)
    if isinstance(vals, list):
        vals = vals[0]
    return np.asarray(vals)


def make_loaders(X_tr, y_tr):
    """Split arrays into train / val DataLoaders."""
    n     = len(X_tr)
    idx   = np.random.RandomState(SEED).permutation(n)
    n_val = max(1, int(n * VAL_FRAC))
    tr_idx, val_idx = idx[n_val:], idx[:n_val]

    def _loader(X, y, shuffle):
        return DataLoader(
            TensorDataset(torch.tensor(X, dtype=torch.float32),
                          torch.tensor(y, dtype=torch.float32)),
            batch_size=BATCH_SIZE,
            shuffle=shuffle,
            generator=g if shuffle else None,
            num_workers=0,
            pin_memory=True,
            drop_last=shuffle,
        )

    return (_loader(X_tr[tr_idx],  y_tr[tr_idx],  shuffle=True),
            _loader(X_tr[val_idx], y_tr[val_idx],  shuffle=False))


# =============================================================================
# MODEL DEFINITION - TabNet
#
# TabNet (Arik & Pfister, 2021) uses sequential attention to select a sparse
# subset of features at each step. Each step produces a partial output; the
# final prediction is the sum across all steps. The attention mechanism is
# penalized via an entropy sparsity loss (LAMBDA_SPARSE) to encourage each
# step to focus on a different subset of features.
#
# Architecture per step:
#   shared FC layers (reused across steps) → step-specific FC layers
#   → split: attention transformer + feature transformer
#   → attentive feature selection (soft mask via sparsemax)
#   → partial output accumulated across steps
#
# Key hyperparameters:
#   N_STEPS:       number of sequential decision steps (depth analog)
#   HIDDEN:        N_a = N_d, shared and step feature dimension
#   GAMMA:         controls how much prior attention suppresses reuse
#   LAMBDA_SPARSE: entropy penalty weight on attention masks
# =============================================================================

class GLULayer(nn.Module):
    """Single Gated Linear Unit layer used inside TabNet feature transformer."""
    def __init__(self, in_dim: int, out_dim: int, fc: nn.Linear = None,
                 momentum: float = 0.02):
        super().__init__()
        self.fc   = fc if fc is not None else nn.Linear(in_dim, out_dim * 2, bias=False)
        self.bn   = nn.BatchNorm1d(out_dim * 2, momentum=momentum)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        h = self.bn(self.fc(x))
        x1, x2 = h.chunk(2, dim=-1)
        return x1 * torch.sigmoid(x2)


class FeatureTransformer(nn.Module):
    """
    Two-layer GLU block: one shared layer (weights reused across steps) +
    one step-specific layer. Each contributes sqrt(0.5) residual scaling
    as in the original TabNet paper.
    """
    def __init__(self, in_dim: int, out_dim: int,
                 shared_fc: nn.Linear, momentum: float = 0.02):
        super().__init__()
        self.shared = GLULayer(in_dim,  out_dim, fc=shared_fc, momentum=momentum)
        self.step   = GLULayer(out_dim, out_dim, momentum=momentum)
        self.scale  = (0.5 ** 0.5)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        x = self.shared(x) * self.scale
        x = (x + self.step(x)) * self.scale
        return x


class AttentiveTransformer(nn.Module):
    """
    Learns which features to attend to at each step given the prior attention
    scales. Uses sparsemax instead of softmax for truly sparse selection.
    GAMMA controls suppression of previously attended features.
    """
    def __init__(self, in_dim: int, n_features: int, momentum: float = 0.02):
        super().__init__()
        self.fc = nn.Linear(in_dim, n_features, bias=False)
        self.bn = nn.BatchNorm1d(n_features, momentum=momentum)

    def forward(self, h: torch.Tensor,
                prior_scales: torch.Tensor) -> torch.Tensor:
        x = self.bn(self.fc(h))
        x = x * prior_scales
        # sparsemax via sorting — produces sparser attention than softmax
        return self._sparsemax(x)

    @staticmethod
    def _sparsemax(z: torch.Tensor) -> torch.Tensor:
        z_sorted, _ = torch.sort(z, dim=-1, descending=True)
        k = torch.arange(1, z.shape[-1] + 1, device=z.device, dtype=z.dtype)
        cumsum = torch.cumsum(z_sorted, dim=-1)
        support = (1 + k * z_sorted) > cumsum
        k_z = support.sum(dim=-1, keepdim=True).float()
        tau = (cumsum.gather(1, (k_z - 1).long().clamp(min=0)) - 1.0) / k_z
        return torch.clamp(z - tau, min=0.0)


class TabNet(nn.Module):
    def __init__(self,
                 in_dim:        int,
                 hidden:        int   = 256,
                 n_steps:       int   = 6,
                 n_shared:      int   = 2,
                 n_indep:       int   = 2,
                 gamma:         float = 1.5,
                 momentum:      float = 0.02,
                 dropout:       float = 0.1):
        super().__init__()

        self.n_steps  = n_steps
        self.gamma    = gamma
        self.in_dim   = in_dim
        self.hidden   = hidden

        # Input BN — normalises raw features before any processing
        self.input_bn = nn.BatchNorm1d(in_dim, momentum=momentum)

        # Shared FC layers reused across all steps (n_shared layers)
        # All share the same weight matrix — reduces parameters, improves
        # generalisation by forcing steps to agree on a common representation.
        self.shared_fcs = nn.ModuleList([
            nn.Linear(in_dim if i == 0 else hidden, hidden * 2, bias=False)
            for i in range(n_shared)
        ])

        # Per-step feature transformers and attentive transformers
        self.feat_transformers = nn.ModuleList()
        self.attn_transformers = nn.ModuleList()

        for step in range(n_steps):
            # First shared FC determines input dim; use shared_fcs[0]
            ft = FeatureTransformer(
                in_dim  = in_dim,
                out_dim = hidden,
                shared_fc = self.shared_fcs[0],
                momentum  = momentum,
            )
            self.feat_transformers.append(ft)
            self.attn_transformers.append(
                AttentiveTransformer(hidden, in_dim, momentum=momentum)
            )

        self.final_fc  = nn.Linear(hidden, 1)
        self.dropout   = nn.Dropout(dropout)

    def forward(self, x: torch.Tensor, return_masks: bool = False):
        B          = x.shape[0]
        x_norm     = self.input_bn(x)

        # Prior scales accumulate attention — suppresses features already used
        prior_scales = torch.ones(B, self.in_dim, device=x.device)
        h            = torch.zeros(B, self.hidden, device=x.device)
        total_entropy = torch.zeros(1, device=x.device)
        output        = torch.zeros(B, self.hidden, device=x.device)
        masks         = []

        for step in range(self.n_steps):
            # Attention mask over input features
            mask = self.attn_transformers[step](h, prior_scales)
            masks.append(mask)

            # Update prior: multiply by gamma * (1 - mask) to suppress reuse
            prior_scales = prior_scales * (self.gamma - mask)

            # Masked input into feature transformer
            x_masked = x_norm * mask
            h = self.feat_transformers[step](x_masked)
            h = self.dropout(h)

            # ReLU activates the step contribution
            output = output + F.relu(h)

            # Sparsity loss: entropy of attention mask
            total_entropy += (-mask * torch.log(mask + 1e-15)).sum(dim=-1).mean()

        # Average step contributions and project to output
        out = self.final_fc(output / self.n_steps)

        if return_masks:
            return out, total_entropy / self.n_steps, masks
        return out, total_entropy / self.n_steps


# =============================================================================
# TRAINING FUNCTION
# =============================================================================

def ML_PyTorch(X_ts, X_tr, y_tr):
    """Full training run with TabNet and sparsity-regularised loss."""
    train_loader, val_loader = make_loaders(X_tr, y_tr)

    in_dim = X_tr.shape[1]

    print(f'Rank {rank}: Training started at {datetime.now()}')
    print(f'Rank {rank}: hidden={HIDDEN}, n_steps={N_STEPS}, '
          f'n_shared={N_SHARED}, n_indep={N_INDEP}, '
          f'gamma={GAMMA}, lambda_sparse={LAMBDA_SPARSE}')
    print(f'Rank {rank}: lr={MAX_LR:.2e}, dropout={DROPOUT:.3f}')
    print(f'Rank {rank}: train={int(len(X_tr) * (1 - VAL_FRAC))}, '
          f'val={int(len(X_tr) * VAL_FRAC)}')

    model = TabNet(
        in_dim   = in_dim,
        hidden   = HIDDEN,
        n_steps  = N_STEPS,
        n_shared = N_SHARED,
        n_indep  = N_INDEP,
        gamma    = GAMMA,
        momentum = MOMENTUM,
        dropout  = DROPOUT,
    ).to(device)

    optimizer = torch.optim.AdamW(model.parameters(),
                                  lr=MAX_LR,
                                  weight_decay=WEIGHT_DECAY,
                                  betas=(0.9, 0.99), eps=1e-7)

    warmup = torch.optim.lr_scheduler.LinearLR(
        optimizer, start_factor=0.1, end_factor=1.0,
        total_iters=WARMUP_EPOCHS)
    cosine = torch.optim.lr_scheduler.CosineAnnealingLR(
        optimizer, T_max=max(1, NUM_EPOCHS - WARMUP_EPOCHS), eta_min=1e-6)
    scheduler = torch.optim.lr_scheduler.SequentialLR(
        optimizer, schedulers=[warmup, cosine], milestones=[WARMUP_EPOCHS])

    criterion = nn.SmoothL1Loss(beta=HUBER_BETA)

    best_val          = float('inf')
    best_state        = None
    epochs_no_improve = 0

    for epoch in tqdm(range(NUM_EPOCHS), desc=f'Rank {rank} Training'):
        # ── train ─────────────────────────────────────────────────────────────
        model.train()
        epoch_loss, n_batches = 0.0, 0
        for Xb, yb in train_loader:
            Xb, yb = Xb.to(device), yb.to(device)
            optimizer.zero_grad()
            with torch.amp.autocast(device_type='cuda', dtype=torch.bfloat16):
                pred, entropy = model(Xb)
                task_loss = criterion(pred.squeeze(), yb.squeeze())
                # Sparsity regularisation: penalise non-sparse attention masks
                loss = task_loss + LAMBDA_SPARSE * entropy
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
            optimizer.step()
            epoch_loss += task_loss.item()
            n_batches  += 1
        scheduler.step()

        # ── validate ──────────────────────────────────────────────────────────
        model.eval()
        val_loss, n_vb = 0.0, 0
        with torch.no_grad():
            for Xb, yb in val_loader:
                Xb, yb = Xb.to(device), yb.to(device)
                with torch.amp.autocast(device_type='cuda', dtype=torch.bfloat16):
                    pred, _ = model(Xb)
                    val_loss += criterion(pred.squeeze(),
                                          yb.squeeze()).item()
                n_vb += 1
        val_loss /= max(1, n_vb)

        if val_loss < best_val - 1e-7:
            best_val          = val_loss
            best_state        = copy.deepcopy(model.state_dict())
            epochs_no_improve = 0
        else:
            epochs_no_improve += 1

        if epoch % 5 == 0 or epoch == NUM_EPOCHS - 1:
            print(f'Rank {rank}, Epoch {epoch:3d}, '
                  f'train={epoch_loss / max(1, n_batches):.6f}, '
                  f'val={val_loss:.6f}, '
                  f'lr={optimizer.param_groups[0]["lr"]:.2e}, '
                  f'patience={epochs_no_improve}/{PATIENCE}')

        if epochs_no_improve >= PATIENCE:
            print(f'Rank {rank}: Early stop at epoch {epoch} '
                  f'(best val={best_val:.6f})')
            break

    if best_state is not None:
        model.load_state_dict(best_state)
        print(f'Rank {rank}: Restored best weights (val={best_val:.6f})')

    print(f'Rank {rank}: Training complete at {datetime.now()}')

    # ── Batched inference ─────────────────────────────────────────────────────
    model.eval()
    predictions = []
    with torch.no_grad():
        torch.cuda.empty_cache()
        for i in range(0, len(X_ts), BATCH_SIZE):
            Xb     = torch.tensor(X_ts[i:i + BATCH_SIZE],
                                  dtype=torch.float32).to(device)
            pred_b, _ = model(Xb)
            predictions.append(pred_b.cpu().numpy())
            del Xb
            torch.cuda.empty_cache()

    pred = np.concatenate(predictions, axis=0).ravel()
    return pred, model


# =============================================================================
# MAIN EXECUTION
# =============================================================================

home       = '/scratch/project/prj-16-pi-kenneth-tobin'
resolution = (f'/scratch/user/aaron.sanchez_tamiu.edu/'
              f'TabNet_{state}_500v1noFELC-20140730-{rank}')
train_file = f'{home}/{state}_trainV6-LC.csv'
test_file  = f'{home}/{state}_testV6-LC.csv'

if rank == 0:
    print(f'\nMPI Multi-Node TabNet Configuration:')
    print(f'  Total ranks  : {size}')
    print(f'  GPUs visible : {_num_gpus}')
    print(f'  State        : {state}')
    print(f'  hidden       : {HIDDEN}')
    print(f'  n_steps      : {N_STEPS}')
    print(f'  n_shared     : {N_SHARED}')
    print(f'  n_indep      : {N_INDEP}')
    print(f'  gamma        : {GAMMA}')
    print(f'  lambda_sparse: {LAMBDA_SPARSE}')
    print(f'  dropout      : {DROPOUT:.3f}')
    print(f'  lr           : {MAX_LR:.2e}')
    print(f'  weight_decay : {WEIGHT_DECAY}')
    print(f'  huber_beta   : {HUBER_BETA:.3f}')
    print(f'  Batch size   : {BATCH_SIZE}')
    print(f'  Loading from : {train_file}')

# ── Column sets ───────────────────────────────────────────────────────────────

all_vars = ['PageName', 'Clay', 'Sand', 'Silt', 'Elevation', 'Slope',
            'Aspect', 'MODIS', 'SMERGE', 'Date', 'LAI', 'ALB', 'Temp',
            'AHRR', 'Month', 'Year', 'LandC']

# noFE: raw integer Month and Year.
# var2 = ['Clay','Sand','Silt','Elevation','Aspect','Slope',   <- [0-5]  static
#          'LAI','MODIS','ALB','Temp','LandC',                 <- [6-10] dynamic
#          'Month','Year']                                      <- [11-12] temporal
var2 = ['Clay', 'Sand', 'Silt', 'Elevation', 'Aspect', 'Slope',
        'LAI', 'MODIS', 'ALB', 'Temp', 'LandC',
        'Month', 'Year']

# ── Load + clean ──────────────────────────────────────────────────────────────

def load_and_clean(path):
    df = pd.read_csv(path, usecols=all_vars, engine='pyarrow').dropna()
    df['Soil'] = df['Sand'] + df['Silt'] + df['Clay']
    df = df[(df['Soil'] >= 0.9999) & (df['Soil'] < 1.0001)].reset_index()
    for col in ['index', 'level_0']:
        if col in df.columns:
            df.drop(columns=[col], inplace=True)
    return df.dropna()


train_df = load_and_clean(train_file)
test_df  = load_and_clean(test_file)

print(f'Rank {rank}: Data loaded — Train={len(train_df)}, Test={len(test_df)}')

# ── Strided per-rank split ────────────────────────────────────────────────────

data_chunk_train = train_df.iloc[rank::size].reset_index(drop=True)
data_chunk_test  = test_df.iloc[rank::size].reset_index(drop=True)

print(f'Rank {rank}: Chunk {rank}/{size} — '
      f'Train={len(data_chunk_train)}, Test={len(data_chunk_test)}')

X_train = data_chunk_train[var2].astype(np.float32).values
y_train = data_chunk_train[['SMERGE']].astype(np.float32).values
X_test  = data_chunk_test[var2].astype(np.float32).values

print(f'Rank {rank}: X_train shape={X_train.shape}')

# =============================================================================
# TRAIN + PREDICT
# =============================================================================

ml_out, trained_model = ML_PyTorch(X_test, X_train, y_train)

# ── Evaluate ──────────────────────────────────────────────────────────────────

data_chunk_test['ML_'] = ml_out
corr, p_value = rank_corr_ave(data_chunk_test, 'AHRR', 'ML_')
print(f'Rank {rank}: Spearman corr={corr:.4f}, p={p_value:.4e}')

# ── Save predictions ──────────────────────────────────────────────────────────

output_path = f'{resolution}_{int(rank)}.csv'
try:
    import dask.dataframe as dd
    ddf = dd.from_pandas(data_chunk_test, npartitions=12)
    ddf.to_csv(output_path, single_file=True, index=False)
    print(f'Rank {rank}: Results saved via Dask -> {output_path}')
except Exception as e:
    print(f'Rank {rank}: Dask failed ({e}), falling back to pandas')
    data_chunk_test.to_csv(output_path, index=False)

# =============================================================================
# SHAP FEATURE IMPORTANCE
# =============================================================================

print(f'Rank {rank}: Computing SHAP values...')
raw_shap = compute_shap_values(trained_model, X_train, X_test, device)
raw_shap = np.squeeze(raw_shap, axis=-1)

feat_importance = np.mean(np.abs(raw_shap), axis=0)
shap_df = (pd.DataFrame({'Feature': var2, 'Mean|SHAP|': feat_importance})
             .sort_values('Mean|SHAP|', ascending=False))

shap_path = f'{resolution}_{int(rank)}_shap_importance.csv'
shap_df.to_csv(shap_path, index=False)
print(f'Rank {rank}: SHAP table -> {shap_path}')
print(shap_df.to_string(index=False))

if rank == 0:
    print('\n' + '=' * 60)
    print('All ranks completed successfully!')
    print('=' * 60)
